In [0]:
dbutils.widgets.text("configId", "8")
configId = dbutils.widgets.get("configId")

dbutils.widgets.text("sourceTypeId", "3")
sourceTypeId = dbutils.widgets.get("sourceTypeId")

dbutils.widgets.text("destinationContainerPath", "raw")
destinationContainerPath = dbutils.widgets.get("destinationContainerPath")

dbutils.widgets.text("destinationFolderPath", "DeviceLogs/<DeviceType>/<DeviceId>/<LoadType>/<YYYYMMDD>/<LogType>")
destinationFolderPath = dbutils.widgets.get("destinationFolderPath")

dbutils.widgets.text("invalidFilePathFormat", "DeviceLogs/InvalidFiles/<DeviceType>/<DeviceId>/<LoadType>/<YYYYMMDD>")
invalidFilePathFormat = dbutils.widgets.get("invalidFilePathFormat")

dbutils.widgets.text("parent_run_id", "1")
pipeLineId = dbutils.widgets.get("parent_run_id")

dbutils.widgets.text("job_id", "1")
jobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("ingestionQueueName", "agncom3ingestionframeworkqueue_usb")
ingestionQueueName = dbutils.widgets.get("ingestionQueueName")

dbutils.widgets.text("createdBy", "LoadLogFiles_COM3")
createdBy = dbutils.widgets.get("createdBy")

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePathPrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")



In [0]:
%run ../Configurations/Init_Scripts

In [0]:
import multiprocessing
from multiprocessing.pool import ThreadPool
from azure.servicebus import ServiceBusClient, ServiceBusMessage, AutoLockRenewer, NEXT_AVAILABLE_SESSION,ServiceBusSubQueue
from azure.servicebus.exceptions import OperationTimeoutError
from azure.servicebus.management import MessageCountDetails
import os
import json
import zipfile
import shutil
from datetime import datetime
import traceback

In [0]:
validLogFileTypes = ['UL','UE','SYS','ML','MLT','ENGR','MEDAPPUI-USERINTERACTION']
# filePathPrefix = '/dbfs/mnt'

In [0]:
def MessageProcessing(message,receiver):
    processedFileList = []
    fileDetails = dict()
    fileDetails['ConfigId'] = configId
    fileDetails['SourceTypeId'] = sourceTypeId
    fileDetails['PipelineRunId'] = pipeLineId
    fileDetails['JobId'] = jobId
    
    try:
        message_json = json.loads(str(message))
        eventType = message_json.get('eventType')
        subject = message_json.get('subject')
        extensionList = subject.lower().split('.')[1:]

        sourceFilePath = subject.replace('/blobServices/default/containers','').replace('/blobs','')
        sourceFilePathList = sourceFilePath.split('/')

        sourceContainerPath = sourceFilePathList[1]
        sourceFolderPath = sourceFilePath[sourceFilePath.find('/',1)+1:sourceFilePath.rfind('/',1)]
        sourceFileName = sourceFilePathList[-1]

        loadType = ''
        if(sourceContainerPath.lower() == 'logs-scanned'):
            loadType = 'IOT'
        elif(sourceContainerPath.lower() == 'usbdecrypted-zip'):
            loadType = 'USB'

        if(sourceFolderPath != ''):
            sourceFilePath = sourceContainerPath+'/'+sourceFolderPath+'/'
        else:
            sourceFilePath = sourceContainerPath+'/'
            
        sourceFileArray = sourceFileName.split('_')
        fileNameStatus = ''
        if('zip' in extensionList):
            if(len(sourceFileArray)>=5):
                deviceId = sourceFileArray[2].upper()
                logDate = sourceFileArray[4][:8]
                fileNameStatus = 'valid'
            else:
                deviceId = sourceFolderPath.upper()
                fileNameStatus = 'invalid'
                
        else:
            deviceId = sourceFolderPath
            fileNameStatus = 'notzipfile'
        if(sourceContainerPath.lower() == 'logs-scanned'):
            fileDetails['SourceFolderPath'] = '/'+sourceFolderPath+'/'
        elif(sourceContainerPath.lower() == 'usbdecrypted-zip'):
            fileDetails['SourceFolderPath'] = '/'+sourceFolderPath
            
        fileDetails['SourceFileName'] = sourceFileName
        fileDetails['SourceContainerPath'] = sourceContainerPath
        fileDetails['DestinationContainerPath'] = destinationContainerPath
        fileDetails['DeviceId'] = deviceId
        fileDetails['DeviceType'] = ''   
        fileDetails['LoadType'] = loadType        

        if(eventType == "Microsoft.Storage.BlobCreated" and sourceContainerPath.lower() in ('usbdecrypted-zip','logs-scanned')):
            if(sourceContainerPath == 'logs-scanned' and sourceFileName.lower().startswith('zlog')):  
                fileDetails['PipelineStatus'] = 'Duplicate'
                fileDetails['ErrorMessage'] = 'zlog file copied to logs container using azure function. Ignore files'
                processedFileList.append(fileDetails)
                logIntoIngestionLogTable(processedFileList,createdBy)
                receiver.complete_message(message)
                return ''
            elif(fileNameStatus == 'invalid'):
                invalidDestinationFilePath = (invalidFilePathFormat.replace('<DeviceId>',deviceId)
                                                                  .replace('/<YYYYMMDD>','')
                                                                  .replace('/<DeviceType>','')
                                                                  .replace('<LoadType>',loadType))

                fileDetails['DestinationFolderPath'] = '/'+invalidDestinationFilePath+'/'
                destinationPath =  destinationContainerPath+'/'+invalidDestinationFilePath+'/'+sourceFileName
                copyFiles(filePathPrefix+'/'+sourceFilePath+sourceFileName,filePathPrefix+'/'+destinationPath)
                fileDetails['PipelineStatus'] = 'InValidFile'
                fileDetails['ErrorMessage']  = 'filename is not following naming standards Invalid zip file {}'.format(subject)   
                processedFileList.append(fileDetails)
                logIntoIngestionLogTable(processedFileList,createdBy)
                (receiver.dead_letter_message(message,
                                              reason=fileDetails['PipelineStatus'],
                                              error_description=fileDetails['ErrorMessage']))
                print(fileDetails['ErrorMessage'])
                return ''
            elif(fileNameStatus == 'notzipfile'):
                fileDetails['PipelineStatus'] = 'InValidFile'
                fileDetails['ErrorMessage'] = 'file extention is not zip file'
                processedFileList.append(fileDetails)
                logIntoIngestionLogTable(processedFileList,createdBy)
                receiver.complete_message(message)
                return ''
            elif(fileNameStatus == 'valid'):
                destinationFilePath = (destinationFolderPath.replace('<DeviceId>',deviceId)
                                                            .replace('<YYYYMMDD>',logDate)
                                                            .replace('<LoadType>',loadType))
                invalidDestinationFilePath = (invalidFilePathFormat.replace('<DeviceId>',deviceId)
                                                                   .replace('<YYYYMMDD>',logDate)
                                                                   .replace('<LoadType>',loadType)) 

                with zipfile.ZipFile(filePathPrefix+'/'+sourceFilePath+sourceFileName) as zip:
                    for zip_info in zip.infolist():
                        if zip_info.filename[-1] == '/':
                            continue
                        zipFileDetails = dict()
                        destinationPath = ''
                        destProcessed = ''
                        stagingFilePath = ''
                        invalidDestProcessed = ''

                        if(sourceContainerPath.lower() == 'logs-scanned'):
                            zipFileDetails['SourceFolderPath'] = '/'+sourceFolderPath+'/'
                        elif(sourceContainerPath.lower() == 'usbdecrypted-zip'):
                            zipFileDetails['SourceFolderPath'] = '/'+sourceFolderPath
                        zipFileDetails['SourceFileName'] = sourceFileName
                        zipFileDetails['SourceContainerPath'] = sourceContainerPath
                        zipFileDetails['DestinationContainerPath'] = destinationContainerPath
                        zipFileDetails['ConfigId'] = configId
                        zipFileDetails['SourceTypeId'] = sourceTypeId
                        zipFileDetails['PipelineRunId'] = pipeLineId
                        zipFileDetails['JobId'] = jobId
                        zipFileDetails['DeviceId'] = deviceId
                        zipFileDetails['SourceFileSize'] = zip_info.file_size
                        zipFileDetails['LoadType'] = loadType

                        osFileName = os.path.basename(zip_info.filename)
                        #override zip info file name to avoid unnecessary folders created in destination
                        zip_info.filename = os.path.basename(zip_info.filename)
                        
                        ziparray = osFileName.split('_')
                        if(len(ziparray)>=3):
                            deviceType = ziparray[0].upper()
                            logType = ziparray[2].upper()

                            zipFileDetails['DeviceType'] = deviceType
                            zipFileDetails['LogType'] = logType

                            if logType in validLogFileTypes  or 'ASSERT' in logType:
                                destinationPath = destinationFilePath.replace('<DeviceType>',deviceType).replace('<LogType>',logType)
                            else:
                                destinationPath = invalidDestinationFilePath.replace('<DeviceType>',deviceType)
                        else:
                            deviceType = ziparray[0].upper()
                            zipFileDetails['DeviceType'] = deviceType                        
                            destinationPath = invalidDestinationFilePath.replace('/<DeviceType>',deviceType)

                        zipFileDetails['DestinationFolderPath'] = '/'+destinationPath+'/'
                        zipFileDetails['DestinationFileName'] = osFileName
                        destinationFullPath = filePathPrefix+'/'+destinationContainerPath+'/'+destinationPath
                        stagingFilePath = filePathPrefix+'/'+destinationContainerPath+'/CopyActivityStaging/'+destinationPath+'/'+sourceFileName

                        try:
                            if(os.path.isfile(destinationFullPath+'/'+osFileName)):
                                destinationFileSize = os.path.getsize(destinationFullPath+'/'+osFileName)
                                sourceFileSize = zip_info.file_size 
                                if(sourceFileSize>destinationFileSize):
                                    copyFiles(destinationFullPath+'/'+osFileName,stagingFilePath+'/BackUp/'+osFileName)
                                    zip.extract(zip_info, stagingFilePath+'/extract')
                                    copyFiles(stagingFilePath+'/extract/'+osFileName,destinationFullPath+'/'+osFileName)
                                    shutil.rmtree(stagingFilePath)
                                    zipFileDetails['PipelineStatus'] = 'Succeeded'
                                    zipFileDetails['ErrorMessage'] = ''                                
                                else:
                                    zipFileDetails['PipelineStatus'] = 'Duplicate'
                                    zipFileDetails['ErrorMessage'] = 'File did not copy as file already exist with samefilesize'
                            else:
                                zip.extract(zip_info, destinationFullPath)
                                zipFileDetails['PipelineStatus'] = 'Succeeded'
                                zipFileDetails['ErrorMessage'] = ''
                        except FileExistsError:
                            zip.extract(zip_info, stagingFilePath+'/extract')
                            copyFiles(stagingFilePath+'/extract/'+osFileName,destinationFullPath+'/'+osFileName)
                            shutil.rmtree(stagingFilePath)
                            zipFileDetails['PipelineStatus'] = 'Succeeded'
                            zipFileDetails['ErrorMessage'] = ''
                        except Exception as e:
                            print(subject)
                            print("{0}: {1}".format(str(e),destinationFullPath))
                            zipFileDetails['PipelineStatus'] = 'Failed'
                            zipFileDetails['ErrorMessage'] = "{0}: {1}".format(str(e),destinationFullPath)
                            traceback.print_exc()
#                             processedFileList.append(zipFileDetails)
#                             logIntoIngestionLogTable(processedFileList,createdBy)
#                             (receiver.dead_letter_message(message,
#                                                   reason=zipFileDetails['PipelineStatus'],
#                                                   error_description=zipFileDetails['ErrorMessage']))
#                             return zipFileDetails['ErrorMessage']
                        finally:
                            processedFileList.append(zipFileDetails)
                logIntoIngestionLogTable(processedFileList,createdBy)
                receiver.complete_message(message)
                return ''
            else:
                fileDetails['PipelineStatus'] = 'UnKnownScenario'
                fileDetails['ErrorMessage'] = 'UnKnownScenario'
                processedFileList.append(fileDetails)
                logIntoIngestionLogTable(processedFileList,createdBy)
                (receiver.dead_letter_message(message,
                                              reason=fileDetails['PipelineStatus'],
                                              error_description=fileDetails['ErrorMessage']))
                return fileDetails['ErrorMessage']
        else:
            receiver.complete_message(message)
            return ''
    except Exception as exp:
        print(subject)
        print(str(exp))
        traceback.print_exc()
        fileDetails['PipelineStatus'] = 'Failed'
        fileDetails['ErrorMessage'] = str(exp)
        processedFileList.append(fileDetails)
        logIntoIngestionLogTable(processedFileList,createdBy)
        (receiver.dead_letter_message(message,
                                      reason=fileDetails['PipelineStatus'],
                                      error_description=fileDetails['ErrorMessage']))
        return fileDetails['ErrorMessage']

In [0]:
def ReceiveMessageIterative(connection_string, queue_name):
    try:
        cpuCount = os.cpu_count()
        messageCount = 10*cpuCount
        concurrancyRun = cpuCount
        renewer = AutoLockRenewer()
        exceptionList = []
        with ServiceBusClient.from_connection_string(connection_string) as sb_client:
            with sb_client.get_queue_receiver(queue_name=queue_name, sub_queue=ServiceBusSubQueue.DEAD_LETTER) as receiver:
                while True:
                    receivedMessages = receiver.receive_messages(max_message_count=messageCount,max_wait_time=5)
                    if(len(receivedMessages)>0):
                        print("Current Time = {}".format(datetime.now()))

                        allMessages = []
                        for message in receivedMessages:
                            renewer.register(receiver, message, max_lock_renewal_duration=6000)
                            allMessages = allMessages + [(message, receiver)]

                        pool = ThreadPool(concurrancyRun)
                        returnList = pool.starmap(MessageProcessing, allMessages)
                        pool.close()

                        for returnValue in returnList:
                            if returnValue:
                                exceptionList.append(returnValue)
                    else:
                        if(exceptionList):
                            print(returnList)
                            raise Exception('Exception raised while processing files and details above')
                        break
                               
        renewer.close()
    except OperationTimeoutError:
        print("If timeout occurs during connecting to a session,It indicates that there might be no non-empty sessions remaining; exiting.This may present as a UserError in the azure portal metric.")
        raise

In [0]:
serviceBusConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="CSServiceBus")
print(ingestionQueueName)
ingestionQueueName = 'agncom3ingestionframeworkqueue_iot'
ReceiveMessageIterative(serviceBusConnectionString, ingestionQueueName)

ingestionQueueName = 'agncom3ingestionframeworkqueue_usb'
ReceiveMessageIterative(serviceBusConnectionString, ingestionQueueName)


